In [69]:
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.stats import binomtest, norm

In [ ]:
_track_type = ["CAB", "CBA", "ACB", "BCA", "ABC", "BAC"]

def load_trial_info(file_path):
    
    mat = sio.loadmat(file_path, squeeze_me=True, struct_as_record=False)
    temp_s = mat["neuro_type_save"]
    data = {name: getattr(temp_s, name) for name in temp_s._fieldnames}
    
    key_list = ["index_correct", "index_miss", "index_false"]
    
    track_type_index = {track_type: [] for track_type in _track_type}
    
    for key in key_list:
        length = data[key].shape[0]
        for i in range(len(_track_type)):
            if i >= length:
                break
            x = data[key][i]
            track_type_index[_track_type[i]].extend(np.asarray(x.index).ravel().tolist())
        
    track_type_index = {
        track_type: np.array(track_type_index[track_type])
        for track_type in _track_type
    }
    
    return track_type_index


def _track_to_abba(track_type: str) -> str:
    s = track_type.replace("C", "")
    if s not in ("AB", "BA"):
        raise ValueError(f"track_type {track_type!r} cannot be mapped to 'AB'/'BA'")
    return s


def _runs_test_binary(seq01: np.ndarray):
    """
    Wald-Wolfowitz runs test for a binary sequence.

    Parameters
    ----------
    seq01 : np.ndarray, shape (n_trials,)
        Binary sequence encoded as 0/1.

    Returns
    -------
    dict
        {
            'n_runs': int,
            'expected_runs': float,
            'z': float,
            'p_two_sided': float
        }
    """
    seq01 = np.asarray(seq01, dtype=int)
    n = len(seq01)
    if n < 2:
        return {
            "n_runs": np.nan,
            "expected_runs": np.nan,
            "z": np.nan,
            "p_two_sided": np.nan,
        }

    n1 = np.sum(seq01 == 1)
    n0 = np.sum(seq01 == 0)

    if n1 == 0 or n0 == 0:
        return {
            "n_runs": 1,
            "expected_runs": np.nan,
            "z": np.nan,
            "p_two_sided": np.nan,
        }

    n_runs = 1 + np.sum(seq01[1:] != seq01[:-1])

    expected = 1 + (2 * n1 * n0) / (n1 + n0)
    var = (
        2 * n1 * n0 * (2 * n1 * n0 - n1 - n0)
        / (((n1 + n0) ** 2) * (n1 + n0 - 1))
    )

    if var <= 0:
        z = np.nan
        p = np.nan
    else:
        z = (n_runs - expected) / np.sqrt(var)
        p = 2 * (1 - norm.cdf(abs(z)))

    return {
        "n_runs": int(n_runs),
        "expected_runs": float(expected),
        "z": float(z) if not np.isnan(z) else np.nan,
        "p_two_sided": float(p) if not np.isnan(p) else np.nan,
    }


def build_ab_ba_sequence(track_type_index: dict, one_based_index: bool = True):
    """
    Reconstruct ordered AB/BA trial sequence from track_type_index.

    Parameters
    ----------
    track_type_index : dict
        Output of load_trial_info(file_path).
        Example:
        {
            'CAB': np.ndarray([...]),
            'CBA': np.ndarray([...]),
            ...
        }

    one_based_index : bool, default=True
        MATLAB trial indices are often 1-based.
        This function will keep the original trial index in 'trial_index_raw'
        and also create a 0-based 'trial_index' for convenience.

    Returns
    -------
    seq_df : pd.DataFrame
        Columns:
        - trial_index_raw
        - trial_index
        - track_type
        - abba
        - code   (AB=1, BA=0)
    """
    rows = []

    for track_type, indices in track_type_index.items():
        abba = _track_to_abba(track_type)
        idx = np.asarray(indices).astype(int).ravel()

        for t in idx:
            rows.append((int(t), track_type, abba))

    if len(rows) == 0:
        raise ValueError("No trials found in track_type_index.")

    seq_df = pd.DataFrame(rows, columns=["trial_index_raw", "track_type", "abba"])
    seq_df = seq_df.sort_values("trial_index_raw").reset_index(drop=True)

    # 检查同一个 trial index 是否被重复分到多个类型
    dup = seq_df["trial_index_raw"].duplicated(keep=False)
    if dup.any():
        dup_df = seq_df.loc[dup].copy()
        raise ValueError(
            "Some trial indices appear in more than one track type.\n"
            f"{dup_df}"
        )

    if one_based_index:
        seq_df["trial_index"] = seq_df["trial_index_raw"] - 1
    else:
        seq_df["trial_index"] = seq_df["trial_index_raw"]

    seq_df["code"] = (seq_df["abba"] == "AB").astype(int)
    return seq_df


def analyze_ab_ba_randomness(track_type_index: dict, one_based_index: bool = True):
    """
    Analyze whether AB/BA trial order is consistent with a random sequence.

    This function checks:
    1) overall balance (AB vs BA)
    2) transition balance:
       AB->AB, AB->BA, BA->AB, BA->BA
    3) repeat vs alternate bias
    4) runs test

    Parameters
    ----------
    track_type_index : dict
        Output of load_trial_info(file_path)

    one_based_index : bool, default=True
        Whether the original indices are MATLAB-style 1-based.

    Returns
    -------
    result : dict
        Main statistics and summary tables.
    """
    seq_df = build_ab_ba_sequence(track_type_index, one_based_index=one_based_index)

    labels = seq_df["abba"].to_numpy()
    codes = seq_df["code"].to_numpy()
    n = len(labels)

    # overall balance
    n_ab = int(np.sum(labels == "AB"))
    n_ba = int(np.sum(labels == "BA"))
    balance_test = binomtest(n_ab, n=n_ab + n_ba, p=0.5)

    # transitions
    if n >= 2:
        prev = labels[:-1]
        curr = labels[1:]

        n_ab_ab = int(np.sum((prev == "AB") & (curr == "AB")))
        n_ab_ba = int(np.sum((prev == "AB") & (curr == "BA")))
        n_ba_ab = int(np.sum((prev == "BA") & (curr == "AB")))
        n_ba_ba = int(np.sum((prev == "BA") & (curr == "BA")))

        n_repeat = n_ab_ab + n_ba_ba
        n_alternate = n_ab_ba + n_ba_ab

        repeat_test = binomtest(n_repeat, n=n_repeat + n_alternate, p=0.5)

        ab_follow_total = n_ab_ab + n_ab_ba
        ba_follow_total = n_ba_ab + n_ba_ba

        p_ab_after_ab = n_ab_ab / ab_follow_total if ab_follow_total > 0 else np.nan
        p_ba_after_ba = n_ba_ba / ba_follow_total if ba_follow_total > 0 else np.nan

        cond_test_ab = (
            binomtest(n_ab_ab, n=ab_follow_total, p=0.5).pvalue
            if ab_follow_total > 0 else np.nan
        )
        cond_test_ba = (
            binomtest(n_ba_ba, n=ba_follow_total, p=0.5).pvalue
            if ba_follow_total > 0 else np.nan
        )

        trans_table = pd.DataFrame(
            [[n_ab_ab, n_ab_ba],
             [n_ba_ab, n_ba_ba]],
            index=["prev_AB", "prev_BA"],
            columns=["curr_AB", "curr_BA"]
        )
    else:
        n_ab_ab = n_ab_ba = n_ba_ab = n_ba_ba = 0
        n_repeat = n_alternate = 0
        repeat_test = None
        p_ab_after_ab = np.nan
        p_ba_after_ba = np.nan
        cond_test_ab = np.nan
        cond_test_ba = np.nan
        trans_table = pd.DataFrame(
            [[0, 0], [0, 0]],
            index=["prev_AB", "prev_BA"],
            columns=["curr_AB", "curr_BA"]
        )

    runs_res = _runs_test_binary(codes)

    summary = {
        "n_trials": int(n),
        "n_AB": n_ab,
        "n_BA": n_ba,
        "p_balance_AB_vs_0.5": float(balance_test.pvalue),

        "AB_to_AB": int(n_ab_ab),
        "AB_to_BA": int(n_ab_ba),
        "BA_to_AB": int(n_ba_ab),
        "BA_to_BA": int(n_ba_ba),

        "n_repeat": int(n_repeat),
        "n_alternate": int(n_alternate),
        "repeat_rate": float(n_repeat / (n_repeat + n_alternate)) if (n_repeat + n_alternate) > 0 else np.nan,
        "p_repeat_vs_0.5": float(repeat_test.pvalue) if repeat_test is not None else np.nan,

        "p_AB_after_AB": float(p_ab_after_ab) if not np.isnan(p_ab_after_ab) else np.nan,
        "p_BA_after_BA": float(p_ba_after_ba) if not np.isnan(p_ba_after_ba) else np.nan,
        "p_conditional_AB_after_AB_vs_0.5": float(cond_test_ab) if not np.isnan(cond_test_ab) else np.nan,
        "p_conditional_BA_after_BA_vs_0.5": float(cond_test_ba) if not np.isnan(cond_test_ba) else np.nan,

        "runs_n": runs_res["n_runs"],
        "runs_expected": runs_res["expected_runs"],
        "runs_z": runs_res["z"],
        "runs_p": runs_res["p_two_sided"],
    }

    result = {
        "sequence_df": seq_df,
        "transition_table": trans_table,
        "summary": summary,
    }
    return result


def print_ab_ba_randomness_report(result: dict, alpha: float = 0.05):
    """
    Pretty-print the result of analyze_ab_ba_randomness.
    """
    s = result["summary"]

    print("=== Overall balance ===")
    print(f"n_trials: {s['n_trials']}")
    print(f"AB: {s['n_AB']}, BA: {s['n_BA']}")
    print(f"balance p (AB vs 0.5): {s['p_balance_AB_vs_0.5']:.6g}")
    if s["p_balance_AB_vs_0.5"] < alpha:
        print("-> overall AB/BA ratio significantly deviates from 1:1")
    else:
        print("-> overall AB/BA ratio is consistent with 1:1")

    print("\n=== Transition table ===")
    print(result["transition_table"])

    print("\n=== Repeat / alternate ===")
    print(f"repeat: {s['n_repeat']}, alternate: {s['n_alternate']}")
    print(f"repeat_rate: {s['repeat_rate']:.4f}")
    print(f"repeat p (vs 0.5): {s['p_repeat_vs_0.5']:.6g}")
    if s["p_repeat_vs_0.5"] < alpha:
        if s["repeat_rate"] > 0.5:
            print("-> significant repeat bias")
        else:
            print("-> significant alternation bias")
    else:
        print("-> no significant repeat/alternation bias")

    print("\n=== Conditional transition tests ===")
    print(f"P(AB | prev=AB): {s['p_AB_after_AB']:.4f}, p={s['p_conditional_AB_after_AB_vs_0.5']:.6g}")
    print(f"P(BA | prev=BA): {s['p_BA_after_BA']:.4f}, p={s['p_conditional_BA_after_BA_vs_0.5']:.6g}")

    print("\n=== Runs test ===")
    print(f"runs_n: {s['runs_n']}")
    print(f"runs_expected: {s['runs_expected']}")
    print(f"runs_z: {s['runs_z']}")
    print(f"runs_p: {s['runs_p']}")
    if not np.isnan(s["runs_p"]):
        if s["runs_p"] < alpha:
            if s["runs_z"] < 0:
                print("-> too few runs: sequence is more clustered than random")
            else:
                print("-> too many runs: sequence alternates more than random")
        else:
            print("-> runs count is consistent with randomness")

In [76]:
file_path = "../../data/HPC_2p/HP02/neuro_type_saveHP02_1_2024-12-23.mat"

In [77]:
track_type_index = load_trial_info(file_path)

res = analyze_ab_ba_randomness(track_type_index, one_based_index=True)
print_ab_ba_randomness_report(res)

seq_df = res["sequence_df"]
seq_df.head()

=== Overall balance ===
n_trials: 124
AB: 67, BA: 57
balance p (AB vs 0.5): 0.419074
-> overall AB/BA ratio is consistent with 1:1

=== Transition table ===
         curr_AB  curr_BA
prev_AB       38       28
prev_BA       28       29

=== Repeat / alternate ===
repeat: 67, alternate: 56
repeat_rate: 0.5447
repeat p (vs 0.5): 0.367297
-> no significant repeat/alternation bias

=== Conditional transition tests ===
P(AB | prev=AB): 0.5758, p=0.267812
P(BA | prev=BA): 0.5088, p=1

=== Runs test ===
runs_n: 57
runs_expected: 62.596774193548384
runs_z: -1.015983380486198
runs_p: 0.309637300812738
-> runs count is consistent with randomness


,trial_index_raw,track_type,abba,trial_index,code
0,1,ABC,AB,0,1
1,2,ABC,AB,1,1
2,3,BAC,BA,2,0
3,4,CAB,AB,3,1
4,5,ABC,AB,4,1


In [73]:
track_type_index

{'CAB': array([  4,  10,  11,  20,  21,  25,  28,  34,  35,  47,  73,  74,  82,
         98, 103, 114, 115, 119, 120,  14]),
 'CBA': array([ 18,  24,  36,  43,  67,  70,  71,  75,  76,  78,  80,  83,  93,
         96, 108, 112, 113, 118, 121,  41]),
 'ACB': array([ 22,  26,  32,  37,  39,  45,  46,  51,  52,  53,  60,  62,  68,
         79,  81,  85,  90,  92, 102, 109, 110,  13,  86]),
 'BCA': array([  9,  12,  19,  49,  54,  56,  61,  65,  66,  69,  72,  87,  94,
        100, 104, 107, 116, 122,  16,  27,  40]),
 'ABC': array([  7,   8,  33,  38,  44,  50,  59,  63,  84,  88,  89,  91,  95,
         99, 101, 106, 111, 117, 123, 124,   5,  15,   1,   2]),
 'BAC': array([ 23,  29,  30,  31,  42,  48,  55,  57,  64,  77,  97, 105,   3,
          6,  17,  58])}